In [1]:
import pandas as pd
import numpy as np

In [2]:

gravity_base = pd.read_csv("../outputs_csv/gravity_base.csv")

/var/folders/rj/tzk4w23s0md7pcnpq4w2fy6c0000gn/T/ipykernel_39404/1761165302.py:1: DtypeWarning: Columns (0: foulNames, 1: foulIds) have mixed types. Specify dtype option on import or set low_memory=False.
  gravity_base = pd.read_csv("../outputs_csv/gravity_base.csv")


In [3]:
gravity_base['dropBackType'].value_counts()

dropBackType
Traditional               5645596
Scramble                  1180300
Unknown                    532224
Designed Rollout Right     259886
Scramble Rollout Right     159126
Designed Rollout Left      142736
Scramble Rollout Left       28578
Designed Run                 4246
Name: count, dtype: int64

All plays with dropBackType = "Unknown" are plays that were nullified by penalty. These plays can be dropped. Screens and RPOs were automatically dropped by Kaggle.

In [5]:
# exclude rollouts, designed runs, and plays nullified by penalty (Unknown)
base_filtered = gravity_base[(gravity_base['dropBackType'] == 'Traditional') | (gravity_base['dropBackType'] == 'Scramble')]

In [6]:
each_play = base_filtered.dropna(subset='event')
each_play=each_play.drop_duplicates(subset=['gameId', 'playId', 'frameID'])
each_play['event'].value_counts()

event
ball_snap                    7417
pass_forward                 6570
autoevent_ballsnap           3279
autoevent_passforward        3250
play_action                  1351
qb_sack                       412
run                           378
pass_arrived                  319
autoevent_passinterrupted     166
man_in_motion                 138
line_set                      121
shift                         101
pass_tipped                    96
first_contact                  68
qb_strip_sack                  57
pass_outcome_incomplete        34
pass_outcome_caught            20
fumble                         12
handoff                        10
fumble_offense_recovered        7
huddle_break_offense            2
tackle                          2
dropped_pass                    1
penalty_flag                    1
Name: count, dtype: int64

In [7]:
each_play = base_filtered.dropna(subset='event')
each_play=each_play.drop_duplicates(subset=['gameId', 'playId', 'frameID'])

# get the frames where the window ends. Window ends when a forward pass is thrown, 
# or when a QB gets sacked, or when a QB gets strip-sacked, or when a QB takes off to run.
end_window_event = each_play[(each_play['event'] == 'pass_forward') | (each_play['event'] == 'qb_sack')
| (each_play['event'] == 'qb_strip_sack') | (each_play['event'] == 'run')][['gameId', 'playId', 'frameID']]
end_window_event = end_window_event.sort_values(by=['gameId', 'playId', 'frameID'])
end_window_event = end_window_event.drop_duplicates(subset=['gameId', 'playId'], keep='first')

# get the frames where the ball is snapped
ball_snaps = each_play[each_play['event'] == 'ball_snap'][['gameId', 'playId','frameID']]
ball_snaps = ball_snaps.drop_duplicates(subset=['gameId', 'playId'], keep='first')

# get all plays where a ball_snapped frame and an end_window frame exists
each_play=each_play.merge(end_window_event[['gameId','playId']], on=['gameId','playId'],how='inner')
each_play=each_play.merge(ball_snaps[['gameId','playId']], on=['gameId','playId'],how='inner')
merged_temp = pd.concat([end_window_event, ball_snaps])
each_play=each_play.merge(merged_temp[['gameId','playId','frameID']], on=['gameId','playId','frameID'], how='inner')

In [8]:
# confirm that each play has two frames: a ball_snap frame and an end_window frame
value_counts = each_play.groupby(['gameId','playId'])['event'].count()
value_counts.value_counts()


event
2    7386
Name: count, dtype: int64

In [9]:
temp = base_filtered.merge(each_play[['gameId','playId']].drop_duplicates(), on=['gameId','playId'], how='inner')

ball_snaps = each_play[each_play['event'] == 'ball_snap']
ball_snaps['frameIdBallSnap'] = ball_snaps['frameID']
end_window_event = each_play[each_play['event'] != 'ball_snap']
end_window_event['frameIdEndWindow'] = end_window_event['frameID']

temp = temp.merge(ball_snaps[['gameId','playId','frameIdBallSnap']], on=['gameId','playId'], how='inner')
temp = temp.merge(end_window_event[['gameId','playId','frameIdEndWindow']], on=['gameId','playId'], how='inner')


temp['three_seconds_after'] = temp['frameIdBallSnap']+30
temp['frameIdEndWindow'] = temp[['frameIdEndWindow', 'three_seconds_after']].min(axis=1)

# filter to an upper bound of the frameIdEndWindow
temp = temp[(temp['frameID'] <= temp['frameIdEndWindow'])]

In [10]:
# FILTER OUT FRAMES WHERE THE QB IS OUTSIDE THE TACKLE BOX

# --------------------------------------------------
# 1. Standardize key columns if needed
# --------------------------------------------------
# Adjust these names if your actual columns differ
GAME_COL = "gameId"
PLAY_COL = "playId"
FRAME_COL = "frameID"
EVENT_COL = "event"
POSITION_COL = "pff_positionLinedUp"
# PLAYER_COL = "displayName"   # or whatever identifies QB if needed
Y_COL = "y"

# # Optional: normalize string columns to avoid case issues
# df[EVENT_COL] = df[EVENT_COL].astype(str).str.lower().str.strip()
# df[POSITION_COL] = df[POSITION_COL].astype(str).str.upper().str.strip()

# --------------------------------------------------
# 2. Get LT y-position at ball_snap for each play
# --------------------------------------------------
lt_snap = (
    temp[
        (temp[EVENT_COL] == "ball_snap") &
        (temp[POSITION_COL] == "LT")
    ][[GAME_COL, PLAY_COL, Y_COL]]
    .drop_duplicates(subset=[GAME_COL, PLAY_COL])
    .rename(columns={Y_COL: "y_LT"})
)

# --------------------------------------------------
# 3. Get RT y-position at ball_snap for each play
# --------------------------------------------------
rt_snap = (
    temp[
        (temp[EVENT_COL] == "ball_snap") &
        (temp[POSITION_COL] == "RT")
    ][[GAME_COL, PLAY_COL, Y_COL]]
    .drop_duplicates(subset=[GAME_COL, PLAY_COL])
    .rename(columns={Y_COL: "y_RT"})
)

# --------------------------------------------------
# 4. Merge y_LT and y_RT onto the full tracking dataframe
# --------------------------------------------------
df_with_tackles = temp.merge(
    lt_snap,
    on=[GAME_COL, PLAY_COL],
    how="inner"
).merge(
    rt_snap,
    on=[GAME_COL, PLAY_COL],
    how="inner"
)

# --------------------------------------------------
# 5. Identify QB rows
# --------------------------------------------------
# Best if you already have a position column for player role on each row.
# Replace this with your actual QB-identifying column if needed.
qb_rows = df_with_tackles[POSITION_COL].astype(str).str.upper().eq("QB")

qb_tracking = df_with_tackles[qb_rows].copy()

# --------------------------------------------------
# 6. Define tackle box boundaries
# --------------------------------------------------
# Since one tackle may have a lower y and the other higher y,
# use min/max to avoid assuming LT < RT or RT < LT.
qb_tracking["tackle_box_min_y"] = qb_tracking[["y_LT", "y_RT"]].min(axis=1)
qb_tracking["tackle_box_max_y"] = qb_tracking[["y_LT", "y_RT"]].max(axis=1)

# --------------------------------------------------
# 7. Keep only QB frames where QB is inside tackle box
# --------------------------------------------------
qb_in_tackle_box = qb_tracking[
    (qb_tracking[Y_COL] >= qb_tracking["tackle_box_min_y"]) &
    (qb_tracking[Y_COL] <= qb_tracking["tackle_box_max_y"])
].copy()

# --------------------------------------------------
# 8. If you want ALL player rows for only those frames where QB is in tackle box
# --------------------------------------------------
valid_qb_frames = qb_in_tackle_box[[GAME_COL, PLAY_COL, FRAME_COL]].drop_duplicates()

tracking_only_when_qb_in_tackle_box = df_with_tackles.merge(
    valid_qb_frames,
    on=[GAME_COL, PLAY_COL, FRAME_COL],
    how="inner"
)

In [11]:
# filter to a lower bound of 5 frames after ball snap
filtered_lower_and_upper_bound = tracking_only_when_qb_in_tackle_box[(tracking_only_when_qb_in_tackle_box['frameID'] >= tracking_only_when_qb_in_tackle_box['frameIdBallSnap']+5)]

max_upper_bound = (
    filtered_lower_and_upper_bound
    .groupby(['gameId', 'playId'])['frameID']
    .max()
    .reset_index()
    .rename(columns={'frameID': 'maxUpperBound'})
)

filtered_lower_and_upper_bound = filtered_lower_and_upper_bound.merge(
    max_upper_bound,
    on=['gameId', 'playId'],
    how='inner'
)
filtered_lower_and_upper_bound['frameIdEndWindow'] = filtered_lower_and_upper_bound['maxUpperBound']
filtered_lower_and_upper_bound = filtered_lower_and_upper_bound.drop(columns=['maxUpperBound'])

In [12]:
# confirm that the longest frame window is 2.5 seconds long.

(filtered_lower_and_upper_bound.groupby(['gameId', 'playId'])['frameID'].max().reset_index() - filtered_lower_and_upper_bound.groupby(['gameId', 'playId'])['frameID'].min().reset_index())['frameID'].max()

np.int64(25)

In [13]:
# filter out plays where the frameIdEndWindow - frameIdBallSnap <= 15

play_windows = (
    filtered_lower_and_upper_bound[['gameId', 'playId', 'frameIdBallSnap', 'frameIdEndWindow']]
    .drop_duplicates()
)

play_windows['window_length'] = (
    play_windows['frameIdEndWindow'] - play_windows['frameIdBallSnap']
)

valid_plays = play_windows[play_windows['window_length'] > 15][['gameId', 'playId']]

tracking_filtered_by_window_length = filtered_lower_and_upper_bound.merge(
    valid_plays,
    on=['gameId', 'playId'],
    how='inner'
)
tracking_filtered_by_window_length

,gameId,playId,season,week,gameDate,quarter,down,yardsToGo,gameClock,play time,...,a,dis,o,dir,event,frameIdBallSnap,frameIdEndWindow,three_seconds_after,y_LT,y_RT
0,2021090900,97,2021,1,09/09/2021,1,3,2,13:33,26:32.100,...,1.61,0.13,119.88,261.95,NaN,6,36,36,26.92,21.31
1,2021090900,97,2021,1,09/09/2021,1,3,2,13:33,26:32.200,...,1.43,0.15,113.65,260.27,NaN,6,36,36,26.92,21.31
2,2021090900,97,2021,1,09/09/2021,1,3,2,13:33,26:32.300,...,1.37,0.19,107.99,261.01,NaN,6,36,36,26.92,21.31
3,2021090900,97,2021,1,09/09/2021,1,3,2,13:33,26:32.400,...,1.63,0.25,114.80,264.15,NaN,6,36,36,26.92,21.31
4,2021090900,97,2021,1,09/09/2021,1,3,2,13:33,26:32.500,...,1.28,0.23,119.55,263.17,NaN,6,36,36,26.92,21.31
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3562587,2021110100,4433,2021,8,11/01/2021,4,4,15,0:35,20:24.400,...,0.28,0.74,48.35,60.83,NaN,7,37,37,26.85,20.70
3562588,2021110100,4433,2021,8,11/01/2021,4,4,15,0:35,20:24.500,...,0.67,0.75,48.35,61.74,NaN,7,37,37,26.85,20.70
3562589,2021110100,4433,2021,8,11/01/2021,4,4,15,0:35,20:24.600,...,1.43,0.75,58.99,63.79,NaN,7,37,37,26.85,20.70
3562590,2021110100,4433,2021,8,11/01/2021,4,4,15,0:35,20:24.700,...,2.01,0.75,63.66,65.69,NaN,7,37,37,26.85,20.70


In [ ]:
# check if i filtered frames correctly


check_df = (
    tracking_filtered_by_window_length
    .groupby(['gameId', 'playId'], as_index=False)['frameID']
    .min()
    .merge(
        tracking_filtered_by_window_length[['gameId', 'playId', 'frameIdBallSnap']].drop_duplicates(),
        on=['gameId', 'playId'],
        how='left'
    )
)

check_df['matches'] = check_df['frameID'] == check_df['frameIdBallSnap'] + 5
print((check_df['matches'] == False).sum())

print((tracking_filtered_by_window_length.groupby(['gameId', 'playId'])['frameID'].max().reset_index() - tracking_filtered_by_window_length.groupby(['gameId', 'playId'])['frameID'].min().reset_index())['frameID'].min())

print((tracking_filtered_by_window_length.groupby(['gameId', 'playId'])['frameID'].max().reset_index() - tracking_filtered_by_window_length.groupby(['gameId', 'playId'])['frameID'].min().reset_index())['frameID'].max())


# check if QB is within tackle box at all times
qbs_only = tracking_filtered_by_window_length[tracking_filtered_by_window_length['pff_positionLinedUp'] == 'QB']
sum_outside_tacklebox = ((
    (qbs_only['y'] >= qbs_only[['y_LT', 'y_RT']].min(axis=1)) &
    (qbs_only['y'] <= qbs_only[['y_LT', 'y_RT']].max(axis=1))
) == False).sum()
print(sum_outside_tacklebox)


0       True
1       True
2       True
3       True
4       True
        ... 
7308    True
7309    True
7310    True
7311    True
7312    True
Name: matches, Length: 7313, dtype: bool

In [ ]:
tracking_filtered_by_window_length.to_csv("../outputs_csv/tracking_filtered_by_play_and_frame.csv", index=False)

In [15]:
base_filtered = base_filtered.merge(
    play_windows[['gameId','playId']],
    on=['gameId','playId'],
    how='inner'
)

base_filtered = base_filtered.merge(
    tracking_filtered_by_window_length.drop_duplicates(['gameId','playId'])[['gameId','playId','frameIdEndWindow']], 
    on=['gameId','playId'],
    how='inner'
)

base_filtered = base_filtered[
    base_filtered['frameID'] <= base_filtered['frameIdEndWindow']
]

base_filtered

,gameId,playId,season,week,gameDate,quarter,down,yardsToGo,gameClock,play time,...,playDirection,x,y,s,a,dis,o,dir,event,frameIdEndWindow
0,2021090900,97,2021,1,09/09/2021,1,3,2,13:33,26:31.100,...,right,37.77,24.22,0.29,0.30,0.03,165.16,84.99,NaN,36
1,2021090900,97,2021,1,09/09/2021,1,3,2,13:33,26:31.200,...,right,37.78,24.22,0.23,0.11,0.02,164.33,92.87,NaN,36
2,2021090900,97,2021,1,09/09/2021,1,3,2,13:33,26:31.300,...,right,37.78,24.24,0.16,0.10,0.01,160.24,68.55,NaN,36
3,2021090900,97,2021,1,09/09/2021,1,3,2,13:33,26:31.400,...,right,37.73,24.25,0.15,0.24,0.06,152.13,296.85,NaN,36
4,2021090900,97,2021,1,09/09/2021,1,3,2,13:33,26:31.500,...,right,37.69,24.26,0.25,0.18,0.04,148.33,287.55,NaN,36
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6739234,2021110100,4433,2021,8,11/01/2021,4,4,15,0:35,20:24.400,...,right,40.36,44.25,7.47,0.28,0.74,48.35,60.83,NaN,37
6739235,2021110100,4433,2021,8,11/01/2021,4,4,15,0:35,20:24.500,...,right,41.02,44.60,7.45,0.67,0.75,48.35,61.74,NaN,37
6739236,2021110100,4433,2021,8,11/01/2021,4,4,15,0:35,20:24.600,...,right,41.68,44.94,7.48,1.43,0.75,58.99,63.79,NaN,37
6739237,2021110100,4433,2021,8,11/01/2021,4,4,15,0:35,20:24.700,...,right,42.36,45.25,7.41,2.01,0.75,63.66,65.69,NaN,37


np.int64(0)

In [21]:
base_filtered.drop_duplicates(subset=['gameId','playId'])

,gameId,playId,season,week,gameDate,quarter,down,yardsToGo,gameClock,play time,...,playDirection,x,y,s,a,dis,o,dir,event,frameIdEndWindow
0,2021090900,97,2021,1,09/09/2021,1,3,2,13:33,26:31.100,...,right,37.77,24.22,0.29,0.30,0.03,165.16,84.99,NaN,36
946,2021090900,137,2021,1,09/09/2021,1,1,10,13:18,28:09.900,...,left,106.92,20.83,0.07,0.08,0.01,153.18,250.65,NaN,32
1760,2021090900,187,2021,1,09/09/2021,1,2,6,12:23,29:15.000,...,left,75.23,28.60,0.27,0.54,0.03,131.50,46.10,line_set,28
2442,2021090900,282,2021,1,09/09/2021,1,1,10,9:56,31:51.600,...,left,48.71,28.12,0.20,0.19,0.03,88.15,43.07,NaN,36
3388,2021090900,349,2021,1,09/09/2021,1,3,15,9:46,34:05.000,...,left,53.50,36.53,0.03,0.03,0.03,85.90,72.33,NaN,33
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6733628,2021110100,4310,2021,8,11/01/2021,4,3,8,1:56,15:52.300,...,left,18.90,47.67,0.00,0.00,0.00,241.60,341.47,NaN,37
6734948,2021110100,4363,2021,8,11/01/2021,4,1,10,1:07,18:40.400,...,right,33.29,26.85,0.00,0.00,0.00,86.74,34.36,NaN,37
6736004,2021110100,4392,2021,8,11/01/2021,4,2,7,1:01,19:17.300,...,right,36.31,20.84,0.00,0.00,0.00,89.16,353.34,NaN,37
6737170,2021110100,4411,2021,8,11/01/2021,4,3,15,0:39,19:39.600,...,right,27.97,20.56,0.01,0.01,0.01,92.81,250.80,NaN,33


In [ ]:
base_filtered.to_csv("../outputs_csv/base_filtered.csv", index=False)

In [17]:
gravity_base.iloc[:,0:20]

,gameId,playId,season,week,gameDate,quarter,down,yardsToGo,gameClock,play time,frameID,possessionTeam,defensiveTeam,yardlineSide,yardlineNumber,absoluteYardlineNumber,offenseFormation,offenseRB,offenseTE,offenseWR
0,2021090900,97,2021,1,09/09/2021,1,3,2,13:33,26:31.100,1,TB,DAL,TB,33,43,SHOTGUN,1.0,1.0,3.0
1,2021090900,97,2021,1,09/09/2021,1,3,2,13:33,26:31.200,2,TB,DAL,TB,33,43,SHOTGUN,1.0,1.0,3.0
2,2021090900,97,2021,1,09/09/2021,1,3,2,13:33,26:31.300,3,TB,DAL,TB,33,43,SHOTGUN,1.0,1.0,3.0
3,2021090900,97,2021,1,09/09/2021,1,3,2,13:33,26:31.400,4,TB,DAL,TB,33,43,SHOTGUN,1.0,1.0,3.0
4,2021090900,97,2021,1,09/09/2021,1,3,2,13:33,26:31.500,5,TB,DAL,TB,33,43,SHOTGUN,1.0,1.0,3.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7952687,2021110100,4433,2021,8,11/01/2021,4,4,15,0:35,20:26.500,54,NYG,KC,NYG,20,30,SHOTGUN,1.0,1.0,3.0
7952688,2021110100,4433,2021,8,11/01/2021,4,4,15,0:35,20:26.600,55,NYG,KC,NYG,20,30,SHOTGUN,1.0,1.0,3.0
7952689,2021110100,4433,2021,8,11/01/2021,4,4,15,0:35,20:26.700,56,NYG,KC,NYG,20,30,SHOTGUN,1.0,1.0,3.0
7952690,2021110100,4433,2021,8,11/01/2021,4,4,15,0:35,20:26.800,57,NYG,KC,NYG,20,30,SHOTGUN,1.0,1.0,3.0
